# 02 · Clean the docs
Turns the raw docs into clean article text for ASP.NET Core 10: keeps only the text for version 10,
inserts include files we have, replaces code pointers with short markers, and removes Markdown noise.
Saves the result to `data/processed/docs_clean.jsonl`.

In [1]:
# ===== CELL 1: Setup =====
# Imports the libraries, sets the folders, and chooses which ASP.NET Core
# version the chatbot should answer for.

import re                          # re: find and replace text patterns
from collections import Counter    # Counter: a dictionary that counts things (every count starts at 0)
from pathlib import Path           # Path: work with folders and file paths

import pandas as pd                # pandas: tables
import yaml                        # yaml: reads the "---" header at the top of each doc (it's written in a format called YAML)

RAW = Path("../data/raw")                 # Folder with the original docs
OUT = Path("../data/processed")           # Folder where the cleaned files will be saved
OUT.mkdir(parents=True, exist_ok=True)    # Create the output folder if it doesn't exist yet (no error if it already does)

TARGET = (10, 0)                   # The version we answer for, ASP.NET Core 10.0, written as (major, minor) so it's easy to compare
TARGET_NAME = "aspnetcore-10.0"    # The same version, written the way Microsoft's docs website writes it (used in links)

stats = Counter()                  # An empty counter; the cleaning functions add to it so we can report what was done

In [2]:
# ===== CELL 2: The version filter =====
# The docs contain blocks like:
#     :::moniker range=">= aspnetcore-8.0 < aspnetcore-9.0"
#     ...text that applies only to those versions...
#     :::moniker-end
# These functions keep the blocks that apply to TARGET (10.0) and delete the rest.

# A lookup table: comparison symbol -> a tiny function that does that comparison.
# "lambda v, t: ..." is a one-line function: give it a version v and the target t, and it returns True or False.
OPS = {
    ">=": lambda v, t: t >= v,     # the target is at least version v
    "<=": lambda v, t: t <= v,     # the target is at most version v
    ">":  lambda v, t: t > v,      # the target is newer than v
    "<":  lambda v, t: t < v,      # the target is older than v
    "=":  lambda v, t: t == v,     # the target is exactly v
    "":   lambda v, t: t == v,     # no symbol written also means "exactly v"
}

CONDITION = re.compile(r"(>=|<=|>|<|=)?\s*aspnetcore-(\d+)\.(\d+)")   # Pattern for one condition like ">= aspnetcore-8.0". It captures: the symbol (optional, "?"), the major number and the minor number (\d+ = one or more digits)

def applies(range_expr, target=TARGET):    # Define a function: does this range text include our target version?
    """True if a moniker range like '>= aspnetcore-8.0 < aspnetcore-9.0' includes the target version."""   # A docstring: a short description of the function
    for option in range_expr.split("||"):              # "||" means OR in these ranges, so split them and check each alternative separately
        conditions = CONDITION.findall(option)         # Find every condition in this alternative, e.g. [(">=", "8", "0"), ("<", "9", "0")]
        if conditions and all(OPS[op]((int(a), int(b)), target) for op, a, b in conditions):   # If there is at least one condition AND every condition is True for the target (int() turns the text "8" into the number 8)...
            return True                                # ...then this range includes our version
    return False                                       # No alternative matched, so the range doesn't include our version

MONIKER = re.compile(                              # Pattern that finds one complete version block
    r'^[ \t]*:::moniker\s+range="([^"]+)"[ \t]*\n'  # The opening line :::moniker range="..."; the range text is captured as group 1
    r'((?:(?!:::moniker\s+range).)*?)'              # The block's content (group 2): as little text as possible, never including another opening line (so blocks inside blocks are handled from the inside out)
    r'^[ \t]*:::moniker-end[ \t]*$\n?',             # The closing line :::moniker-end
    re.DOTALL | re.MULTILINE,                      # DOTALL: "." may also match line breaks; MULTILINE: "^" and "$" mean the start/end of each line
)

def resolve_monikers(text):                        # Define a function that cleans all version blocks in a text
    """Keep version blocks that apply to TARGET, delete the rest."""
    while True:                                    # Repeat until nothing changes (this handles blocks inside blocks)
        new = MONIKER.sub(lambda m: m.group(2) if applies(m.group(1)) else "", text)   # Replace every block with its content if its range applies, or with nothing if it doesn't
        if new == text:                            # If this pass changed nothing...
            return new                             # ...we're done, so return the cleaned text
        text = new                                 # Otherwise continue with the new text and go round again

# Quick self-test: check the function on examples where we know the right answer
print(applies(">= aspnetcore-8.0"))                     # 10.0 is at least 8.0      -> should print True
print(applies(">= aspnetcore-8.0 < aspnetcore-9.0"))    # 10.0 is not below 9.0     -> False
print(applies(">= aspnetcore-11.0"))                    # 10.0 is not at least 11.0 -> False
print(applies("< aspnetcore-6.0"))                      # 10.0 is not below 6.0     -> False

True
False
False
False


In [3]:
# ===== CELL 3: Headers and includes =====
# strip_yaml():       separates the "---" header (title, description...) from the article text.
# resolve_includes(): replaces each [!INCLUDE[...](path)] line with the text of that file,
#                     or removes the line if we don't have the file.

def strip_yaml(text):                                       # Define a function that takes a document's full text
    """Split the '---' header from the body. Returns (metadata dict, body)."""
    m = re.match(r"\A---\s*\n(.*?)\n---\s*\n", text, re.DOTALL)   # Look for "---" at the very start, then the header text (captured), then a closing "---"
    if not m:                                               # If there's no header...
        return {}, text                                     # ...return an empty dictionary and the text unchanged
    try:                                                    # "try" = attempt something that might fail
        meta = yaml.safe_load(m.group(1))                   # Read the header as YAML, giving a dictionary like {"title": "...", "description": "..."}
    except yaml.YAMLError:                                  # If the header is badly formatted and can't be read...
        meta = None                                         # ...treat it as missing
    return (meta if isinstance(meta, dict) else {}), text[m.end():]   # Return the dictionary (or {} if it isn't one) and the text after the header (m.end() = where the header ended)

INCLUDE = re.compile(r"\[!INCLUDE\s*\[[^\]]*\]\(([^)]+)\)\]")   # Pattern for [!INCLUDE[anything](PATH)]; the PATH is captured as group 1

def resolve_path(ref, current_file):                        # Turn an include path into a real file location on disk
    ref = ref.split("?")[0].split("#")[0]                   # Remove anything after "?" or "#" (extra options we don't need)
    if ref.startswith("~/"):                                # "~/" means "starting from the docs root folder"...
        return RAW / ref[2:]                                # ...so join data/raw with the rest of the path (ref[2:] skips the first 2 characters, "~/")
    return (current_file.parent / ref).resolve()            # Otherwise the path is relative to the current file's folder; resolve() works out the full path

def resolve_includes(text, current_file, depth=0):          # Replace the include lines in "text"; depth counts how many includes deep we are
    """Insert included files we have; drop the ones we don't."""
    def replace(m):                                         # A helper that decides what ONE include line becomes (m = the match that was found)
        target = resolve_path(m.group(1), current_file)     # Work out where the included file is
        if depth >= 3 or not target.is_file():              # If we're already 3 levels deep (a safety limit) or the file doesn't exist on disk...
            stats["includes dropped (file not available)"] += 1   # ...count it as dropped...
            return ""                                       # ...and replace the include line with nothing
        stats["includes inserted"] += 1                     # Otherwise, count it as inserted
        inner = target.read_text(encoding="utf-8-sig").replace("\r\n", "\n")   # Read the included file, turning Windows line breaks (\r\n) into normal ones (\n)
        inner = resolve_monikers(strip_yaml(inner)[1])      # Drop its header (keep only the body, item [1]) and apply the version filter to it
        return resolve_includes(inner, target, depth + 1)   # Included files can contain includes too, so process them the same way, one level deeper
    return INCLUDE.sub(replace, text)                       # Find every include line in the text and replace it using the helper above

In [4]:
# ===== CELL 4: Markdown cleanup =====
# Removes or simplifies Markdown that is just noise for search: code pointers,
# comments, images, links, API references and callout markers.

CODE_DIRECTIVE = re.compile(r':::code\b[^\n]*?source="([^"]+)"[^\n]*?:::')   # Pattern for :::code ... source="PATH" ... ::: on one line; the PATH is captured
CODE_LEGACY = re.compile(r"\[!code-[\w-]+\[[^\]]*\]\(([^)]+)\)\]")           # Pattern for the older style [!code-csharp[...](PATH)]; the PATH is captured

def code_marker(m):                                        # Decide what a code pointer becomes
    stats["code samples replaced with a marker"] += 1      # Count it
    name = m.group(1).split("?")[0].split("#")[0].rstrip("/").split("/")[-1]   # From the path: drop options after "?" or "#", then keep only the file name (the part after the last "/")
    return f"[Code sample: {name}]"                        # Replace the pointer with a short, readable marker, e.g. [Code sample: Program.cs]

def xref_name(m):                                          # Decide what an <xref:...> API reference becomes
    """<xref:Microsoft.AspNetCore.Components.RenderTreeBuilder> -> RenderTreeBuilder"""
    uid = m.group(1).split("?")[0]                         # The reference text, without any options after "?"
    uid = re.sub(r"%2[Aa]|\*", "", uid)                    # Remove "%2A" or "*" (a code meaning "all overloads of this method")
    uid = re.sub(r"(%60|`)\d+", "", uid)                   # Remove "`1" or "%601" (a code meaning "a generic type with 1 type parameter")
    if "/" in uid:                                         # If it contains "/", it's a link to another article, not an API...
        return f"(see {uid})"                              # ...so keep it as a readable "(see blazor/components/index)"
    return uid.split("(")[0].rstrip(".").split(".")[-1]    # Otherwise drop any "(parameters)" and keep only the last part after the final "." (the short name)

def clean_markdown(text):                                  # Apply all the cleanup rules to one article's text, in order
    text = CODE_DIRECTIVE.sub(code_marker, text)           # :::code pointers   -> [Code sample: ...]
    text = CODE_LEGACY.sub(code_marker, text)              # [!code-...] pointers -> [Code sample: ...]
    text = re.sub(r"<!--.*?-->", "", text, flags=re.DOTALL)               # Delete HTML comments <!-- ... --> (notes for the doc writers, not for readers)
    text = re.sub(r":::image\b[^\n]*?:::", "", text)                      # Delete :::image ... ::: lines
    text = re.sub(r"!\[[^\]]*\]\([^)]*\)", "", text)                       # Delete Markdown images ![alt text](file.png)
    text = re.sub(r"\[([^\]]+)\]\(xref:[^)]*\)", r"\1", text)              # [text](xref:...) -> just "text" (\1 = the captured text)
    text = re.sub(r"<xref:([^>]+)>", xref_name, text)                      # <xref:...> -> the short API name, using xref_name() above
    text = re.sub(r"\[([^\]]+)\]\([^)]*\)", r"\1", text)                   # Any other link [text](url) -> just "text"
    text = re.sub(r"^>\s*\[!(NOTE|TIP|IMPORTANT|WARNING|CAUTION)\]\s*$",   # A callout line like "> [!WARNING]"...
                  lambda m: m.group(1).title() + ":",                      # ...becomes "Warning:" (.title() turns "WARNING" into "Warning")
                  text, flags=re.MULTILINE)
    text = re.sub(r"^>[ \t]?", "", text, flags=re.MULTILINE)               # Remove the ">" at the start of quoted lines
    text = re.sub(r"^[ \t]*:::[^\n]*$", "", text, flags=re.MULTILINE)      # Delete any remaining lines starting with ":::" (zone, row, column markers...)
    text = re.sub(r"\n{3,}", "\n\n", text)                                # Replace 3 or more line breaks in a row with just 2 (one blank line)
    return text.strip()                                                    # Remove blank space at the very start and end, and return the result

In [5]:
# ===== CELL 5: Run the cleaner on every file =====
# Goes through all 495 files, skips the ones that aren't real articles,
# cleans the rest and saves them to data/processed/docs_clean.jsonl.

records = []          # An empty list; one dictionary per kept article goes in here
raw_chars = 0         # Running total of the original size of the kept articles (to report how much we removed)

for p in sorted(RAW.rglob("*.md")):                 # Loop over every .md file in alphabetical order (so the result is the same every run)
    rel = p.relative_to(RAW).as_posix()             # The file's path inside data/raw, e.g. "blazor/advanced-scenarios.md"

    if "/includes/" in rel or "/samples/" in rel:   # Include fragments and sample READMEs aren't articles...
        stats["skipped: include fragment or sample README"] += 1   # ...so count them...
        continue                                    # ...and skip to the next file

    raw = p.read_text(encoding="utf-8-sig").replace("\r\n", "\n")   # Read the file and normalise the line breaks
    meta, body = strip_yaml(raw)                    # Split it into the header dictionary and the article text

    if not meta.get("title"):                       # No title in the header means it isn't a proper article...
        stats["skipped: no title"] += 1             # ...so count it...
        continue                                    # ...and skip it
    if meta.get("monikerRange") and not applies(str(meta["monikerRange"])):   # If the WHOLE page is marked for certain versions and 10.0 isn't one of them...
        stats["skipped: whole page is for older versions"] += 1             # ...count it...
        continue                                                            # ...and skip it

    body = resolve_monikers(body)                   # Step 1: keep only the text for version 10.0
    body = resolve_includes(body, p)                # Step 2: insert the include files we have, drop the rest
    body = clean_markdown(body)                     # Step 3: remove Markdown noise

    if len(body) < 300:                             # If almost nothing is left after cleaning...
        stats["skipped: too short after cleaning"] += 1   # ...count it...
        continue                                    # ...and skip it

    page = rel[:-3]                                 # The page name without ".md" (drop the last 3 characters)
    url_path = page[:-len("index")] if page.endswith("/index") else page   # On the website, ".../index" pages are just ".../", so remove "index" from those names
    raw_chars += len(raw)                           # Add this article's original size to the running total
    records.append({                                # Save everything we know about this article
        "doc_id": page,                             # A unique name for the article, e.g. "blazor/advanced-scenarios"
        "area": rel.split("/")[0],                  # "blazor", "fundamentals" or "mvc" (the first folder)
        "title": meta["title"],                     # The article title from the header
        "description": meta.get("description", ""), # The short description from the header ("" if there isn't one)
        "url": f"https://learn.microsoft.com/aspnet/core/{url_path}?view={TARGET_NAME}",   # The link to this page on Microsoft's site, for version 10.0
        "text": body,                               # The cleaned article text
        "chars": len(body),                         # Its size after cleaning
    })

clean = pd.DataFrame(records)                       # Turn the list into a table
clean.to_json(OUT / "docs_clean.jsonl", orient="records", lines=True, force_ascii=False)   # Save it with one article per line in JSON format (JSONL); force_ascii=False keeps characters like "é" readable

print(f"Articles kept: {len(clean)}")               # How many articles survived
print(f"Size: {raw_chars:,} → {clean['chars'].sum():,} characters "      # Total size before -> after ({:,} adds thousands separators)
      f"({1 - clean['chars'].sum() / raw_chars:.0%} removed)\n")        # ...and the percentage removed ({:.0%} shows a fraction as a whole percentage)
for k, v in stats.most_common():                    # Loop over everything we counted, biggest count first
    print(f"  {k}: {v}")                            # Print each count
print()                                             # Print an empty line
print(clean.groupby("area")["chars"].agg(["count", "sum", "mean"]).round(0))   # Per area: number of articles, total size and average size

Articles kept: 324
Size: 6,746,076 → 4,443,982 characters (34% removed)

  code samples replaced with a marker: 1265
  includes dropped (file not available): 412
  skipped: include fragment or sample README: 160
  includes inserted: 141
  skipped: whole page is for older versions: 10
  skipped: too short after cleaning: 1

              count      sum     mean
area                                 
blazor          152  2586658  17017.0
fundamentals    128  1390160  10861.0
mvc              44   467164  10617.0


In [6]:
# ===== CELL 6: Check that nothing messy is left =====
# Searches the cleaned text for the patterns we meant to remove.
# Every count should be 0.

leftovers = {                        # Name -> pattern for each kind of mess we removed
    ":::moniker": r":::moniker",     # version blocks
    "[!INCLUDE": r"\[!INCLUDE",      # include lines
    "<xref:": r"<xref:",             # API references
    ":::code": r":::code",           # new-style code pointers
    "[!code-": r"\[!code-",          # old-style code pointers
}
for name, pat in leftovers.items():  # Check each pattern in turn
    print(f"{name:12s} still in {clean['text'].str.contains(pat, regex=True).sum()} docs")   # Count the cleaned docs that still contain it ({name:12s} pads the name to 12 characters so the lines up neatly)

:::moniker   still in 0 docs
[!INCLUDE    still in 0 docs
<xref:       still in 0 docs
:::code      still in 0 docs
[!code-      still in 0 docs


In [7]:
# ===== CELL 7: Before and after for the same page =====
# Prints the cleaned version of the page we looked at in 01_explore,
# so we can compare it with the raw version.

doc = clean[clean["doc_id"] == "blazor/advanced-scenarios"].iloc[0]   # Find the row for this article and take it
print(doc["title"], "\n", doc["url"])   # Print its title and link ("\n" = a new line)
print("=" * 80)                         # A separator line
print(doc["text"][:2500])               # The first 2,500 characters of the cleaned text

ASP.NET Core Blazor advanced scenarios (render tree construction) 
 https://learn.microsoft.com/aspnet/core/blazor/advanced-scenarios?view=aspnetcore-10.0
# ASP.NET Core Blazor advanced scenarios (render tree construction)

This article describes the advanced scenario for building Blazor render trees manually with RenderTreeBuilder.

Use of RenderTreeBuilder to create components is an *advanced scenario*. A malformed component (for example, an unclosed markup tag) can result in undefined behavior. Undefined behavior includes broken content rendering, loss of app features, and ***compromised security***.

## Manually build a render tree (`RenderTreeBuilder`)

RenderTreeBuilder provides methods for manipulating components and elements, including building components manually in C# code.

Consider the following `PetDetails` component, which can be manually rendered in another component.

`PetDetails.razor`:

[Code sample: PetDetails.razor]

In the following `BuiltContent` component, the lo